# H&M 2년 CLV 이중축 M2 — seed 42 validation

60일에서 동결한 모델 구조와 `TARGET_RHO=0.2`를 H&M 전체기간에 적용합니다. M1과 `dual_clv_fixed`만 실행하며 대조군·sweep·seed 43/44·test·holdout은 실행하지 않습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys

drive.mount('/content/drive')
REVIEWED_SHA = '4976cd43504a25dd4a6526720fd91a0be7dfc829'
REPO_DIR = Path('/content/clv-m2-lightgcn-runner-hm2y')
os.chdir('/content')
shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('검토된 코드:', actual_sha)


In [ ]:
import json, torch
from lightgcn_clv_dual_hm2y_seed42 import TARGET_RHO, configure_hm2y_seed42, preflight_summary, run_hm2y_seed42

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
ROOT = Path('/content/drive/MyDrive/논문/data')
cfg = configure_hm2y_seed42(
    out_dir=str(ROOT / 'results_clv_dual_hm_2y_seed42'),
    m1_checkpoint_dir=str(ROOT / 'results_v3_hm'),
)
assert TARGET_RHO == 0.2 and cfg.window_days is None
assert cfg.seed_list == (42,) and not cfg.eval_test and not cfg.eval_holdout
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


## 실행

이 셀은 H&M 전체기간 M1 준비와 dual-axis 주모형 학습을 시작합니다. 완료되면 Drive에 checkpoint와 validation 결과가 저장됩니다.


In [ ]:
result_df = run_hm2y_seed42(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

display(result_df.sort_values(['model_id']))
print('운영점:', result_df.attrs['operating_point'])
print('seed 42 판정:', result_df.attrs['decision'])
display(pd.read_csv(result_df.attrs['result_paths']['delta_csv']))
print('결과 파일:', result_df.attrs['result_paths'])
print('완료: 결과 검토 전에는 seed 43·44, 대조군, test를 실행하지 않습니다.')
